In [ ]:
"""
Thu thập dữ liệu thời tiết & chất lượng không khí từ Open-Meteo API
Các cột đầu ra: Local_Time, UTC_Time, City, Country_Code, Timezone,
                AQI, AQI_Pollutant, CO, NO2, O3, PM10, PM25, SO2,
                Clouds, Precipitation, Pressure, Relative_Humidity,
                Temperature, Wind_Speed, Wind_Direction, BLH

──────────────────────────────────────────────
 HƯỚNG DẪN CHẠY TRÊN GOOGLE COLAB
──────────────────────────────────────────────
▌CELL 1 — Cài thư viện (chỉ cần chạy 1 lần)
  !pip install -q requests pandas numpy pytz

▌CELL 2 — (Tuỳ chọn) Mount Google Drive
  from google.colab import drive
  drive.mount('/content/drive')
  # Đổi OUTPUT_FILE thành: "/content/drive/MyDrive/weather_aqi_hourly.csv"

▌CELL 3 — Dán toàn bộ code này vào và chạy
──────────────────────────────────────────────
"""

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import pytz
import time
import os
import sys

# ─────────────────────────────────────────────
#  CẤU HÌNH
# ─────────────────────────────────────────────
LOCATIONS = [
    {"city": "Hanoi", "country_code": "VN", "lat": 21.0285, "lon": 105.8542, "timezone": "Asia/Ho_Chi_Minh"},
]

AIR_QUALITY_START = "2022-08-04"
WEATHER_START     = "2022-08-04"
END_DATE          = datetime.now(timezone.utc).strftime("%Y-%m-%d")

OUTPUT_FILE = "/content/weather_aqi_hourly.csv"

# ─────────────────────────────────────────────
#  TÍNH AQI — US EPA breakpoints
# ─────────────────────────────────────────────

def linear(i_lo, i_hi, c_lo, c_hi, c):
    return round((i_hi - i_lo) / (c_hi - c_lo) * (c - c_lo) + i_lo)

def aqi_pm25(c):
    if c is None or np.isnan(c): return None
    c = round(c, 1)
    bp = [(0,12.0,0,50),(12.1,35.4,51,100),(35.5,55.4,101,150),
          (55.5,150.4,151,200),(150.5,250.4,201,300),(250.5,350.4,301,400),(350.5,500.4,401,500)]
    for c_lo, c_hi, i_lo, i_hi in bp:
        if c_lo <= c <= c_hi:
            return linear(i_lo, i_hi, c_lo, c_hi, c)
    return 500 if c > 500.4 else 0

def aqi_pm10(c):
    if c is None or np.isnan(c): return None
    c = int(c)
    bp = [(0,54,0,50),(55,154,51,100),(155,254,101,150),
          (255,354,151,200),(355,424,201,300),(425,504,301,400),(505,604,401,500)]
    for c_lo, c_hi, i_lo, i_hi in bp:
        if c_lo <= c <= c_hi:
            return linear(i_lo, i_hi, c_lo, c_hi, c)
    return 500 if c > 604 else 0

def aqi_o3(c):
    if c is None or np.isnan(c): return None
    c_ppb = c / 1.9957
    bp = [(0,54,0,50),(55,70,51,100),(71,85,101,150),
          (86,105,151,200),(106,200,201,300)]
    for c_lo, c_hi, i_lo, i_hi in bp:
        if c_lo <= c_ppb <= c_hi:
            return linear(i_lo, i_hi, c_lo, c_hi, c_ppb)
    return 300 if c_ppb > 200 else 0

def aqi_no2(c):
    if c is None or np.isnan(c): return None
    c_ppb = c / 1.8816
    bp = [(0,53,0,50),(54,100,51,100),(101,360,101,150),
          (361,649,151,200),(650,1249,201,300),(1250,1649,301,400),(1650,2049,401,500)]
    for c_lo, c_hi, i_lo, i_hi in bp:
        if c_lo <= c_ppb <= c_hi:
            return linear(i_lo, i_hi, c_lo, c_hi, c_ppb)
    return 500 if c_ppb > 2049 else 0

def aqi_co(c):
    if c is None or np.isnan(c): return None
    c_ppm = c / 1145.0
    bp = [(0,4.4,0,50),(4.5,9.4,51,100),(9.5,12.4,101,150),
          (12.5,15.4,151,200),(15.5,30.4,201,300),(30.5,40.4,301,400),(40.5,50.4,401,500)]
    for c_lo, c_hi, i_lo, i_hi in bp:
        if c_lo <= c_ppm <= c_hi:
            return linear(i_lo, i_hi, c_lo, c_hi, c_ppm)
    return 500 if c_ppm > 50.4 else 0

def aqi_so2(c):
    if c is None or np.isnan(c): return None
    c_ppb = c / 2.6196
    bp = [(0,35,0,50),(36,75,51,100),(76,185,101,150),
          (186,304,151,200),(305,604,201,300),(605,804,301,400),(805,1004,401,500)]
    for c_lo, c_hi, i_lo, i_hi in bp:
        if c_lo <= c_ppb <= c_hi:
            return linear(i_lo, i_hi, c_lo, c_hi, c_ppb)
    return 500 if c_ppb > 1004 else 0

AQI_FUNCS = {
    "PM2.5": aqi_pm25, "PM10": aqi_pm10,
    "O3": aqi_o3,      "NO2": aqi_no2,
    "CO": aqi_co,      "SO2": aqi_so2,
}

def compute_overall_aqi(row):
    results = {}
    col_map = {"PM2.5": "PM25", "PM10": "PM10", "O3": "O3",
               "NO2": "NO2", "CO": "CO", "SO2": "SO2"}
    for pollutant, func in AQI_FUNCS.items():
        val = row.get(col_map[pollutant])
        try:
            sub = func(float(val)) if val is not None and not pd.isna(val) else None
        except (TypeError, ValueError):
            sub = None
        if sub is not None:
            results[pollutant] = sub
    if not results:
        return None, None
    dominant = max(results, key=results.get)
    return results[dominant], dominant

# ─────────────────────────────────────────────
#  GỌI API
# ─────────────────────────────────────────────

def fetch_air_quality(lat, lon, timezone_str, start, end):
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat, "longitude": lon,
        "hourly": "pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone",
        "timezone": timezone_str,
        "start_date": start, "end_date": end,
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    return r.json()

WEATHER_VARS = (
    "temperature_2m,relative_humidity_2m,precipitation,"
    "surface_pressure,cloud_cover,wind_speed_10m,uv_index,"
    "wind_direction_10m,boundary_layer_height"
)

def fetch_weather(lat, lon, timezone_str, start, end):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "hourly": WEATHER_VARS,
        "timezone": timezone_str,
        "start_date": start, "end_date": end,
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    return r.json()

def fetch_weather_recent(lat, lon, timezone_str):
    """Bù gap ~7 ngày gần nhất mà archive chưa có."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, "longitude": lon,
        "hourly": WEATHER_VARS,
        "timezone": timezone_str,
        "past_days": 7,
        "forecast_days": 1,
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

# ─────────────────────────────────────────────
#  XỬ LÝ DỮ LIỆU
# ─────────────────────────────────────────────

def json_to_df(data):
    hourly = data.get("hourly", {})
    df = pd.DataFrame(hourly)
    df.rename(columns={"time": "datetime_local"}, inplace=True)
    df["datetime_local"] = pd.to_datetime(df["datetime_local"])
    return df

def process_location(loc):
    city         = loc["city"]
    country_code = loc["country_code"]
    lat          = loc["lat"]
    lon          = loc["lon"]
    tz_str       = loc["timezone"]

    print(f"\n{'='*60}")
    print(f"  Đang xử lý: {city} ({country_code})")
    print(f"{'='*60}")

    # ── 1. Air Quality ──
    print(f"  [1/3] Fetching air quality {AIR_QUALITY_START} → {END_DATE} ...")
    aq_data = fetch_air_quality(lat, lon, tz_str, AIR_QUALITY_START, END_DATE)
    df_aq = json_to_df(aq_data)
    df_aq.rename(columns={
        "pm2_5": "PM25", "pm10": "PM10",
        "carbon_monoxide": "CO", "nitrogen_dioxide": "NO2",
        "sulphur_dioxide": "SO2", "ozone": "O3",
    }, inplace=True)
    print(f"     → {len(df_aq):,} hàng")

    # ── 2. Weather Archive ──
    print(f"  [2/3] Fetching weather archive {WEATHER_START} → {END_DATE} ...")
    archive_end = (datetime.now(timezone.utc) - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
    weath_data = fetch_weather(lat, lon, tz_str, WEATHER_START, archive_end)
    df_w = json_to_df(weath_data)
    df_w.rename(columns={
        "temperature_2m":        "Temperature",
        "relative_humidity_2m":  "Relative_Humidity",
        "precipitation":         "Precipitation",
        "surface_pressure":      "Pressure",
        "cloud_cover":           "Clouds",
        "wind_speed_10m":        "Wind_Speed",
        "wind_direction_10m":    "Wind_Direction",
        "boundary_layer_height": "BLH",
    }, inplace=True)
    # Loại bỏ UV Index (không nằm trong danh sách cột cần)
    df_w.drop(columns=["uv_index"], errors="ignore", inplace=True)
    print(f"     → {len(df_w):,} hàng (archive)")

    # ── 3. Weather Recent (bù gap) ──
    print(f"  [3/3] Fetching recent weather (past 7 days) ...")
    try:
        recent_data = fetch_weather_recent(lat, lon, tz_str)
        df_recent = json_to_df(recent_data)
        df_recent.rename(columns={
            "temperature_2m":        "Temperature",
            "relative_humidity_2m":  "Relative_Humidity",
            "precipitation":         "Precipitation",
            "surface_pressure":      "Pressure",
            "cloud_cover":           "Clouds",
            "wind_speed_10m":        "Wind_Speed",
            "wind_direction_10m":    "Wind_Direction",
            "boundary_layer_height": "BLH",
        }, inplace=True)
        df_recent.drop(columns=["uv_index"], errors="ignore", inplace=True)
        df_w = pd.concat([df_w, df_recent]).drop_duplicates("datetime_local", keep="first")
        print(f"     → {len(df_recent):,} hàng (recent), tổng: {len(df_w):,}")
    except Exception as e:
        print(f"     ⚠ Bỏ qua recent: {e}")

    # ── 4. Merge weather + AQ ──
    df = pd.merge(df_w, df_aq, on="datetime_local", how="inner")
    df.sort_values("datetime_local", inplace=True)
    df.reset_index(drop=True, inplace=True)

    # ── 5. Thêm cột thời gian ──
    tz_obj = pytz.timezone(tz_str)
    df["Local_Time"] = df["datetime_local"].dt.tz_localize(tz_obj, ambiguous="NaT", nonexistent="NaT")

    # ── 6. Tính AQI ──
    print("  Tính AQI ...")
    aqi_values, aqi_pollutants = [], []
    for _, row in df.iterrows():
        val, pol = compute_overall_aqi(row)
        aqi_values.append(val)
        aqi_pollutants.append(pol)
    df["AQI"]           = aqi_values
    df["AQI_Pollutant"] = aqi_pollutants

    # ── 8. Chỉ giữ đúng các cột cần thiết ──
    final_cols = [
        "Local_Time",
        "AQI", "AQI_Pollutant",
        "CO", "NO2", "O3", "PM10", "PM25", "SO2",
        "Clouds", "Precipitation", "Pressure",
        "Relative_Humidity", "Temperature", "Wind_Speed",
        "Wind_Direction", "BLH",
    ]
    # Chỉ lấy các cột tồn tại trong df
    final_cols = [c for c in final_cols if c in df.columns]
    df = df[final_cols]

    print(f"  ✔ Hoàn thành {city}: {len(df):,} hàng | "
          f"{df['Local_Time'].min()} → {df['Local_Time'].max()}")
    print(f"  Tổng cột: {len(df.columns)}")
    return df


# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────

def main():
    all_dfs = []
    for loc in LOCATIONS:
        try:
            df = process_location(loc)
            all_dfs.append(df)
            time.sleep(1)
        except Exception as e:
            print(f"  ✗ Lỗi {loc['city']}: {e}")

    if not all_dfs:
        print("Không có dữ liệu nào được thu thập.")
        sys.exit(1)

    final = pd.concat(all_dfs, ignore_index=True)

    # Format datetime
    final["Local_Time"] = final["Local_Time"].dt.tz_localize(None).dt.strftime("%Y-%m-%d %H:%M:%S")

    # Output path
    if os.path.isabs(OUTPUT_FILE):
        out_path = OUTPUT_FILE
    else:
        try:
            base = os.path.dirname(os.path.abspath(__file__))
        except NameError:
            base = os.getcwd()
        out_path = os.path.join(base, OUTPUT_FILE)

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    final.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"\n{'='*60}")
    print(f"  ✅ Đã lưu: {out_path}")
    print(f"  Tổng hàng : {len(final):,}")
    print(f"  Cột       : {list(final.columns)}")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    main()




  Đang xử lý: Hanoi (VN)
  [1/3] Fetching air quality 2022-08-04 → 2026-04-22 ...
     → 32,592 hàng
  [2/3] Fetching weather archive 2022-08-04 → 2026-04-22 ...
     → 32,448 hàng (archive)
  [3/3] Fetching recent weather (past 7 days) ...
     → 192 hàng (recent), tổng: 32,592
  Tính AQI ...
  ✔ Hoàn thành Hanoi: 32,592 hàng | 2022-08-04 00:00:00+07:00 → 2026-04-22 23:00:00+07:00
  Tổng cột: 17

  ✅ Đã lưu: /content/weather_aqi_hourly.csv
  Tổng hàng : 32,592
  Cột       : ['Local_Time', 'AQI', 'AQI_Pollutant', 'CO', 'NO2', 'O3', 'PM10', 'PM25', 'SO2', 'Clouds', 'Precipitation', 'Pressure', 'Relative_Humidity', 'Temperature', 'Wind_Speed', 'Wind_Direction', 'BLH']

